# AQT IBEX Q1: running on trapped-ion hardware

> **Billing notice:** IBEX Q1 is real quantum hardware, billed at **35 credits per job plus 2.35 credits per shot**. This notebook submits two jobs of 100 shots each, about **540 credits** in total. The cost cell below works out the price before anything is submitted.

**IBEX Q1** is [Alpine Quantum Technologies](https://www.aqt.eu)' 12-qubit trapped-ion quantum
computer in Innsbruck, Austria. Its qubits are calcium-40 ions, and every qubit can interact
directly with every other, so circuits never need SWAP gates. qBraid connects to it directly
through AQT's cloud, as `aqt:aqt:qpu:ibex-q1`, with no AQT account required.

This notebook covers:

1. Checking the device and its availability
2. Working out a job's cost before submitting it
3. Running a Bell state, and reading the results
4. A 12-qubit GHZ state across the whole register
5. Coming back to a job later, by its ID

In [ ]:
%%capture

# "%%capture" hides the install output; comment it out if you need to debug this step.
%pip install 'qbraid[qiskit,visualization]'

## Connect to the device

`QbraidProvider` picks up your API key from `~/.qbraid/qbraidrc` or the `QBRAID_API_KEY`
environment variable. Inside qBraid Lab it is already configured for you.

In [ ]:
from qbraid.runtime import QbraidProvider

provider = QbraidProvider()
device = provider.get_device("aqt:aqt:qpu:ibex-q1")

device.metadata()

IBEX Q1 takes OpenQASM 2 or 3, and qBraid transpiles to it for you, so a Qiskit, Cirq, Braket or
pytket circuit can go straight to `run()`. It is then compiled to IBEX Q1's native gates: `R`
(a rotation about an axis in the XY plane), `RZ`, and `RXX` (the two-qubit Mølmer–Sørensen
interaction).

### Availability

IBEX Q1 runs during AQT's scheduled **access windows**, not around the clock. `status()` reports
`ONLINE` inside a window. You can submit at any time: a job submitted outside a window waits in
the queue and runs when the next window opens.

In [ ]:
from qbraid.runtime import DeviceStatus

status = device.status()
print(status)

if status != DeviceStatus.ONLINE:
    print("Outside an access window: jobs will queue and run when the next window opens.")

## What a job costs

The device's pricing is part of its metadata. A job costs a fixed amount per task plus an amount
per shot, so you can work out the price of a run before submitting it:

In [ ]:
pricing = device.metadata()["pricing"]


def job_cost(shots: int) -> float:
    """Credits charged for one job of `shots` shots."""
    return pricing["perTask"] + pricing["perShot"] * shots


for shots in (100, 500, 2000):
    print(f"{shots:>5} shots: {job_cost(shots):8.2f} credits")

A job takes at most **2,000 shots**, and a circuit at most **2,000 operations**, on up to
**12 qubits**. Because the per-task price dominates small runs, 100 shots is a good size for a
first experiment.

## A Bell state

Two qubits, entangled with a Hadamard and a CNOT. Ideally, every shot reads `00` or `11`, in
roughly equal numbers:

In [ ]:
from qiskit import QuantumCircuit

bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
bell.measure_all()

bell.draw()

In [ ]:
bell_job = device.run(bell, shots=100)
print(bell_job.id)

`run()` returns as soon as the job is queued. Keep the job ID: if the device is between windows,
the job may not run for a while, and the ID lets you come back to it (see the last section).

`result()` waits for the job to finish, then returns its counts:

In [ ]:
from qbraid.visualization import plot_histogram

bell_counts = bell_job.result().data.get_counts()
print(bell_counts)

plot_histogram(bell_counts)

Most shots land on `00` and `11`. The few that don't are the hardware's real noise: gate errors
and readout errors, which a simulator would only model. Keys are little-endian: **qubit 0 is the
rightmost bit**.

## A 12-qubit GHZ state

All-to-all connectivity means a CNOT can join any two qubits directly. Here we entangle all 12
qubits from qubit 0, a fan-out that would need SWAP gates on a nearest-neighbour chip:

In [ ]:
ghz = QuantumCircuit(12)
ghz.h(0)
for target in range(1, 12):
    ghz.cx(0, target)
ghz.measure_all()

ghz_job = device.run(ghz, shots=100)
ghz_counts = ghz_job.result().data.get_counts()

ghz_population = (ghz_counts.get("0" * 12, 0) + ghz_counts.get("1" * 12, 0)) / 100
print(f"{len(ghz_counts)} distinct outcomes; {ghz_population:.0%} of shots all-zeros or all-ones")

An ideal 12-qubit GHZ state gives only two outcomes. On hardware, eleven two-qubit gates and
twelve readouts each add a little error, so some shots spread to other bitstrings. The fraction
on all-zeros or all-ones is a quick measure of how well the device holds a large entangled state.

## Coming back to a job later

A job ID is all you need to reconnect to a job, from a new session or another machine. This is
useful when a job is waiting for the next access window:

In [ ]:
from qbraid.runtime import QbraidJob

job = QbraidJob(bell_job.id)

print(job.status())
print(job.result().data.get_counts())

## Summary

- IBEX Q1 is AQT's 12-qubit, fully connected trapped-ion computer, available directly on `QbraidProvider` as `aqt:aqt:qpu:ibex-q1`.
- It runs in scheduled access windows. Jobs submitted outside one queue until the next opens.
- A job costs `perTask + perShot × shots` credits, read from `device.metadata()["pricing"]`.
- Jobs take up to 2,000 shots, and circuits up to 2,000 operations on 12 qubits.
- `QbraidJob(job_id)` reconnects to a job later.

<div class="alert alert-block alert-info">
<b>Copyright Notice:</b> 
    All rights reserved © [2026] qBraid. This notebook is part of the qBraid-Lab-Demo repository.
The qBraid-Lab-Demo is licensed under the Apache License, Version 2.0.
You may obtain a copy of the License at <https://www.apache.org/licenses/LICENSE-2.0>.
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
</div>